# ⚡ VRP Factor Model — Real-Time Tick Stream (Alpaca)

Live tick-by-tick variance risk premium model via Alpaca WebSocket.

**Before running:**
1. Open `.env` in this folder
2. Replace `PASTE_YOUR_KEY_ID_HERE` with your Alpaca API Key
3. Replace `PASTE_YOUR_SECRET_KEY_HERE` with your Alpaca Secret Key
4. Save `.env`, then run all cells

```
VRP = IV² − RV²
IV²  = Parkinson high-low estimator (no options data needed)
RV²  = rolling 21-bar realised variance from tick prices
```

In [5]:
# ── CELL 1 · Install packages ──────────────────────────────────────────────
import subprocess, sys

for pkg in ['alpaca-py', 'python-dotenv', 'pandas', 'numpy', 'matplotlib', 'scipy', 'statsmodels']:
    print(f'Installing {pkg}...')
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'],
                       capture_output=True, text=True)
    print('  ✓' if r.returncode == 0 else f'  ⚠️  {r.stderr[-120:]}')

print('\n✅ Done — fill in .env then run Cell 2')

Installing alpaca-py...
  ✓
Installing python-dotenv...
  ✓
Installing pandas...
  ✓
Installing numpy...
  ✓
Installing matplotlib...
  ✓
Installing scipy...
  ✓
Installing statsmodels...
  ✓

✅ Done — fill in .env then run Cell 2


In [6]:
# ── CELL 2 · Load keys from .env ──────────────────────────────────────────
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env file in the same folder

API_KEY    = os.getenv('ALPACA_API_KEY')
SECRET_KEY = os.getenv('ALPACA_SECRET_KEY')

if not API_KEY or 'PASTE' in str(API_KEY):
    raise ValueError(
        '\n❌  Keys not set.'
        '\n    Open the .env file in this folder and fill in your keys:'
        '\n    ALPACA_API_KEY=your_key_here'
        '\n    ALPACA_SECRET_KEY=your_secret_here'
    )

print(f'✓ Keys loaded  (key starts with: {API_KEY[:6]}...)')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from collections import defaultdict, deque
from datetime import datetime
import threading, time, warnings, datetime as dt
warnings.filterwarnings('ignore')

from alpaca.data.historical import StockHistoricalDataClient
from alpaca.data.live import StockDataStream
from alpaca.data.requests import StockBarsRequest
from alpaca.data.timeframe import TimeFrame

SYMBOLS = [
    'AAPL','MSFT','NVDA','GOOGL','META',
    'AMZN','TSLA','JPM','BAC','GS',
    'XOM','JNJ','UNH','WMT','HD'
]

RV_WINDOW = 21
TICK_KEEP = 1000
REFRESH_S = 30

hist_client = StockHistoricalDataClient(API_KEY, SECRET_KEY)
print(f'✓ Connected  |  {len(SYMBOLS)} symbols  |  refresh every {REFRESH_S}s')

ValueError: 
❌  Keys not set.
    Open the .env file in this folder and fill in your keys:
    ALPACA_API_KEY=your_key_here
    ALPACA_SECRET_KEY=your_secret_here

In [ ]:
# ── CELL 3 · Seed history + VRP functions ─────────────────────────────────
print('Seeding 90 days of historical bars...')

req = StockBarsRequest(
    symbol_or_symbols=SYMBOLS,
    timeframe=TimeFrame.Day,
    start=dt.datetime.now() - dt.timedelta(days=90),
    end=dt.datetime.now() - dt.timedelta(days=1),
)
bars = hist_client.get_stock_bars(req).df

price_history = {s: deque(maxlen=TICK_KEEP) for s in SYMBOLS}
hl_history    = {s: deque(maxlen=TICK_KEEP) for s in SYMBOLS}

for sym in SYMBOLS:
    try:
        sym_bars = bars.xs(sym, level='symbol').sort_index()
        price_history[sym].extend(sym_bars['close'].tolist())
        hl_history[sym].extend(zip(sym_bars['high'].tolist(), sym_bars['low'].tolist()))
    except KeyError:
        print(f'  ⚠️  No history for {sym}')

print(f'✓ Seeded {sum(len(v) for v in price_history.values())} bars')


def compute_rv2(prices):
    if len(prices) < RV_WINDOW + 1:
        return np.nan
    p = np.array(list(prices)[-(RV_WINDOW+1):])
    lr = np.diff(np.log(p))
    return float(np.sum(lr**2) * (252 / RV_WINDOW))


def compute_iv2(hl_pairs, window=5):
    if len(hl_pairs) < window:
        return np.nan
    vals = [(np.log(h/l))**2 / (4*np.log(2))
            for h, l in list(hl_pairs)[-window:] if h > l > 0]
    return float(np.mean(vals) * 252) if vals else np.nan


def get_snapshot():
    rows = []
    for sym in SYMBOLS:
        iv2 = compute_iv2(hl_history[sym])
        rv2 = compute_rv2(price_history[sym])
        if np.isnan(iv2) or np.isnan(rv2) or len(price_history[sym]) < 2:
            continue
        rows.append({'symbol': sym,
                     'price': list(price_history[sym])[-1],
                     'iv2': iv2, 'rv2': rv2, 'vrp': iv2 - rv2})
    return pd.DataFrame(rows).set_index('symbol') if rows else pd.DataFrame()


snap = get_snapshot()
print(f'\n✓ Initial VRP snapshot:')
print(snap.sort_values('vrp', ascending=False)[['price','iv2','rv2','vrp']].round(4).to_string())

In [ ]:
# ── CELL 4 · Start tick stream + live dashboard ────────────────────────────
from IPython.display import clear_output

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': '#F8F8F6',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.alpha': 0.25, 'font.family': 'monospace', 'font.size': 10,
})

session_time     = []
session_mean_vrp = []
session_pct_pos  = []
tick_counts      = defaultdict(int)


async def on_trade(trade):
    sym = trade.symbol
    if sym not in SYMBOLS:
        return
    price = float(trade.price)
    prev  = list(price_history[sym])[-1] if price_history[sym] else price
    price_history[sym].append(price)
    h, l = max(price, prev), min(price, prev)
    if h > l:
        hl_history[sym].append((h, l))
    tick_counts[sym] += 1


stream = StockDataStream(API_KEY, SECRET_KEY)
stream.subscribe_trades(on_trade, *SYMBOLS)
threading.Thread(target=stream.run, daemon=True).start()
time.sleep(2)
print('✓ WebSocket connected — receiving ticks')
print(f'Dashboard refreshes every {REFRESH_S}s  ·  Press ■ Stop to quit\n')


def draw():
    snap = get_snapshot()
    if snap.empty:
        print('Waiting for data...'); return

    now = datetime.now()
    session_time.append(now)
    session_mean_vrp.append(snap['vrp'].mean())
    session_pct_pos.append((snap['vrp'] > 0).mean() * 100)
    total_ticks = sum(tick_counts.values())

    clear_output(wait=True)
    fig = plt.figure(figsize=(14, 8))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

    # VRP bar chart
    ax1 = fig.add_subplot(gs[0, :])
    ss  = snap.sort_values('vrp')
    ax1.barh(ss.index, ss['vrp'],
             color=['#3B6D11' if v>0 else '#A32D2D' for v in ss['vrp']], height=0.6)
    ax1.axvline(0, color='#444441', linewidth=0.8)
    ax1.set_title(f'Live VRP  ·  {now.strftime("%H:%M:%S")}  ·  {total_ticks:,} ticks', fontsize=11)
    ax1.set_xlabel('VRP = IV² − RV²')
    for _, (sym, row) in enumerate(ss.iterrows()):
        ax1.text(row['vrp'] + 0.0003 * np.sign(row['vrp']),
                 list(ss.index).index(sym),
                 f'${row["price"]:.2f}', va='center', fontsize=8, color='#5F5E5A')

    # Session mean VRP
    ax2 = fig.add_subplot(gs[1, 0])
    if len(session_time) > 1:
        ax2.plot(session_time, session_mean_vrp, color='#185FA5', linewidth=1.5, marker='o', markersize=3)
        ax2.fill_between(session_time, 0, session_mean_vrp,
                         where=[v>=0 for v in session_mean_vrp], color='#3B6D11', alpha=0.2)
        ax2.fill_between(session_time, 0, session_mean_vrp,
                         where=[v<0  for v in session_mean_vrp], color='#A32D2D', alpha=0.2)
    ax2.axhline(0, color='#444441', linewidth=0.6)
    ax2.set_title('Session mean VRP'); ax2.set_ylabel('VRP')
    plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30, ha='right', fontsize=8)

    # Tick counts
    ax3 = fig.add_subplot(gs[1, 1])
    tc_syms = list(tick_counts.keys())
    ax3.barh(tc_syms, [tick_counts[s] for s in tc_syms], color='#BA7517', height=0.6)
    ax3.set_title('Ticks received (session)'); ax3.set_xlabel('Count')

    # IV² vs RV² scatter
    ax4 = fig.add_subplot(gs[1, 2])
    ax4.scatter(snap['rv2'], snap['iv2'], color='#185FA5', s=55, zorder=3)
    for sym, row in snap.iterrows():
        ax4.annotate(sym, (row['rv2'], row['iv2']),
                     fontsize=7, color='#5F5E5A', xytext=(3,3), textcoords='offset points')
    lim = max(snap[['iv2','rv2']].max().max()*1.1, 0.01)
    ax4.plot([0,lim],[0,lim], color='#A32D2D', linewidth=0.8, linestyle='--', label='VRP=0')
    ax4.set_xlabel('RV²'); ax4.set_ylabel('IV²'); ax4.set_title('IV² vs RV²'); ax4.legend(fontsize=8)

    plt.suptitle('VRP Factor Model — Alpaca Real-Time Ticks', fontsize=12, y=1.01)
    plt.savefig('vrp_live.png', dpi=120, bbox_inches='tight')
    plt.show()

    # Signal table
    print(f'\n{"Symbol":<8} {"Price":>8} {"VRP":>8}  Signal')
    print('─'*40)
    for sym, row in snap.sort_values('vrp', ascending=False).iterrows():
        sig = '🟢 LONG ' if row['vrp']>0.01 else ('🔴 SHORT' if row['vrp']<-0.01 else '⚪ FLAT ')
        print(f'{sym:<8} {row["price"]:>8.2f} {row["vrp"]:>+8.4f}  {sig}')
    print(f'\nNext refresh in {REFRESH_S}s  ·  Press ■ Stop to disconnect')


try:
    while True:
        draw()
        time.sleep(REFRESH_S)
except KeyboardInterrupt:
    stream.stop()
    print('\n⏹ Disconnected.')